### TF-IDF 백터화
- TF(단어 빈도) : 특정 문서 안에서 단어가 얼마나 자주 등장하는가?
- IDF(역문서 빈도) : 전체 문서들 증 그 단어가 등장한 문서의 비율의 역수
- 단순하게 단어의 개수를 세는 방법(CountVectorizer)을 보완하여 이 문서에서는 유독 자주 나오는데, 다른 문서들에서는 잘 안나오는 유독 튀는 단어에 가중치를 부여하는 알고리즘
- 수식적으로 표현하면 TF * IDF로 계산되어 텍스트를 실수형 벡터로 변환
- 매개변수
    - stop_words
        - 기본값 : None
        - 불용어 처리
        - 리스트의 형태를 통해서 특정 단어들을 제외
        -'english'를 인자로 사용하면 영문자를 제외
    - min_df
        - 기본값 : 1
        - 최소 문서 빈도
        - 정수 (개수), 실수(비율)로 설정
        - min_df=3이면 전체 문서에서 3개 미만의 문서에서 등장하는 단어는 '오타'나 '너무 희귀한 단어로'로 간주하여 단어 사전에서는 제외
    - max_df
        - 기본값 : 1.0
        - 최대 문서 빈도
        - 너무 자주 등장하는 단어를 제외
        - max_df = 0.9로 설정하면 전체 문서의 90% 이상 등장하는 단어는 제외
    - ngram_range
        - 기본값 : (1, 1)
        - 단어의 묶음 범위를 지정
        - 만약 (1, 2)로 설정을 하게 되면 1개의 단어(unigram)뿐만이 아니라 2개의 단어묶음(bigram)도 하나의 feature로 사용 (예 : '자연어', '처리',' '자연어 처리')
    - lowercase
        - 기본값 : True
        - 영문 텍스트를 학습할 때, 모든 문자를 소문자로 변환할지의 여부를 결정
    - max_features
        - 기본값 : None
        - 단어 사전에서 단어들의 개수를 지정
        - 중요도(빈도) 기준 상위 n개의 키워드만 사용

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
news_docs=[
    '한국은행 기준 금리 인상 발표 기자 무단전재',
    '미국 연준 금리 인상 가능성 시사 기자 무단전재',
    '한국은행 기준 금리 동결 결정 경제 전망 기자 무단전제',
    '주식 시장 하락 금리 인상 여파 기자',
    '딥러닝 기반 자연어 처리 금리 예측 모델 개발'
]

In [9]:
#불용어 처리
stop_words=['기자','무단전재']

tfidf_vec=TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0,
    lowercase=False,
    max_features=20
)

In [10]:
tfidf_matrix=tfidf_vec.fit_transform(news_docs)

In [11]:
#학습된 단어 사전을 가져오기 --> 데이터프레임의 feature로 사용
feature_names=tfidf_vec.get_feature_names_out()
feature_names


array(['가능성', '가능성 시사', '개발', '결정', '결정 경제', '경제', '경제 전망', '금리', '금리 동결',
       '금리 인상', '기반', '기준', '기준 금리', '동결 결정', '딥러닝', '딥러닝 기반', '모델', '인상',
       '한국은행', '한국은행 기준'], dtype=object)

In [12]:
df=pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
df

,가능성,가능성 시사,개발,결정,결정 경제,경제,경제 전망,금리,금리 동결,금리 인상,기반,기준,기준 금리,동결 결정,딥러닝,딥러닝 기반,모델,인상,한국은행,한국은행 기준
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.246800,0.000000,0.346868,0.000000,0.417868,0.417868,0.000000,0.000000,0.000000,0.000000,0.346868,0.417868,0.417868
1,0.565768,0.565768,0.000000,0.000000,0.000000,0.000000,0.000000,0.269592,0.000000,0.378902,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.378902,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.336513,0.336513,0.336513,0.336513,0.160350,0.336513,0.000000,0.000000,0.271497,0.271497,0.336513,0.000000,0.000000,0.000000,0.000000,0.271497,0.271497
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.449436,0.000000,0.631667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.631667,0.000000,0.000000
4,0.000000,0.000000,0.437393,0.000000,0.000000,0.000000,0.000000,0.208420,0.000000,0.000000,0.437393,0.000000,0.000000,0.000000,0.437393,0.437393,0.437393,0.000000,0.000000,0.000000


In [13]:
#벡터화 + 토큰화
#토큰화 : komoran을 사용
from konlpy.tag import Komoran

In [21]:
#komoran class 생성
komoran=Komoran()

#토큰화 함수를 생성
def tokenize(text):
    #사용할 품사 지정
    allow_pos=['NNP','NNG','VV','VA','SL','MAG']
    stop_words=['기자','무단','전재']
    #글자 수 제한
    len_word=2
    result=[]
    for word, pos in komoran.pos(text):
        if (pos in allow_pos) & (word not in stop_words) & (len(word) >= len_word):
                result.append(word)
    return result

In [24]:
for word in news_docs:
    print(tokenize(word))

['한국은행', '기준', '금리', '인상', '발표']
['미국', '연준', '금리', '인상', '시사']
['한국은행', '기준', '금리', '동결', '결정', '경제', '전망', '전제']
['주식 시장', '하락', '금리', '인상', '여파']
['기반', '자연어', '처리', '금리', '예측', '모델', '개발']


In [25]:
tfidf_vec2=TfidfVectorizer(
    tokenizer=tokenize,
    min_df=1,
    max_df=0.9,
    ngram_range=(1, 1),
    max_features=20
)

In [26]:
tfidf_matrix2=tfidf_vec2.fit_transform(news_docs)
feature_names2=tfidf_vec2.get_feature_names_out()
df2=pd.DataFrame(tfidf_matrix2.toarray(), columns=feature_names2)
df2

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,개발,결정,경제,기반,기준,동결,모델,미국,발표,시사,여파,연준,예측,인상,자연어,전망,전제,주식 시장,처리,한국은행
0,0.000000,0.000000,0.000000,0.000000,0.486484,0.000000,0.000000,0.000000,0.602985,0.000000,0.00000,0.000000,0.000000,0.403826,0.000000,0.000000,0.000000,0.00000,0.000000,0.486484
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.538498,0.000000,0.538498,0.00000,0.538498,0.000000,0.360638,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
2,0.000000,0.398352,0.398352,0.000000,0.321388,0.398352,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.398352,0.398352,0.00000,0.000000,0.321388
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.63907,0.000000,0.000000,0.427993,0.000000,0.000000,0.000000,0.63907,0.000000,0.000000
4,0.408248,0.000000,0.000000,0.408248,0.000000,0.000000,0.408248,0.000000,0.000000,0.000000,0.00000,0.000000,0.408248,0.000000,0.408248,0.000000,0.000000,0.00000,0.408248,0.000000
